# XSD to Synthetic - Library

Core framework classes for generating synthetic data from XSD schemas.

**Import in your main notebook:**
```python
%run "./XSD to Synthetic - Library"
```

## Why Split Library + Main?

- **Library (this file)**: Reusable class definitions, no execution
- **Main notebook**: Configuration + execution only

This keeps the main notebook clean (~10 cells) while making classes reusable.


In [ ]:
# Install required dependencies
%pip install xmlschema dbldatagen faker --quiet

# Restart Python to use newly installed packages
dbutils.library.restartPython()

## Imports

In [ ]:
# Core imports
from __future__ import annotations
import os
import re
import warnings
from dataclasses import dataclass, field
from enum import Enum
from typing import Optional, Any

# External dependencies
import xmlschema
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.types import (
    DataType, StructType, StructField, StringType, IntegerType, 
    LongType, ShortType, ByteType, FloatType, DoubleType, 
    DecimalType, BooleanType, DateType, TimestampType, ArrayType
)

print("Imports loaded successfully")


In [ ]:
# =============================================================================
# Logging Setup (Minimal)
# =============================================================================
import logging

# Default: quiet (warnings only)
_logger = logging.getLogger('xsd_synthetic')
_logger.setLevel(logging.WARNING)
logging.basicConfig(format='%(name)s - %(levelname)s - %(message)s')

def enable_debug():
    """Enable verbose debug output for troubleshooting."""
    logging.getLogger('xsd_synthetic').setLevel(logging.DEBUG)
    print("Debug logging enabled")

def get_logger(name: str) -> logging.Logger:
    """Get a child logger."""
    return logging.getLogger(f'xsd_synthetic.{name}')

print("Logging ready (default: WARNING, call enable_debug() for verbose output)")


In [ ]:
# Cell intentionally left empty (logging consolidated in previous cell)

## TypeConstraints

In [ ]:
# =============================================================================
# TypeConstraints: Holds all XSD restrictions for a type
# =============================================================================
@dataclass
class TypeConstraints:
    """
    Holds all XSD facet restrictions extracted from a type definition.
    Used to generate realistic synthetic data within the schema's bounds.
    """
    base_type: str                              # The primitive XSD type (string, integer, etc.)
    spark_type: DataType = field(default_factory=StringType)
    xsd_type: Optional[Any] = None              # NEW: Original XSD type for sample generation
    
    # Value constraints
    min_value: Optional[float] = None           # minInclusive
    max_value: Optional[float] = None           # maxInclusive
    min_exclusive: Optional[float] = None       # minExclusive
    max_exclusive: Optional[float] = None       # maxExclusive
    
    # String length constraints
    min_length: Optional[int] = None
    max_length: Optional[int] = None
    length: Optional[int] = None                # Exact length
    
    # Pattern and enumeration
    pattern: Optional[str] = None               # Regex pattern
    enumeration: Optional[list[str]] = None     # Allowed values
    
    # Numeric precision
    total_digits: Optional[int] = None
    fraction_digits: Optional[int] = None
    
    # Array occurrence constraints
    min_occurs: Optional[int] = None            # Minimum array elements (from element's minOccurs)
    max_occurs: Optional[int] = None            # Maximum array elements (from element's maxOccurs)
    
    # Fixed value (XSD fixed attribute - value must be exactly this)
    fixed_value: Optional[str] = None           # XSD fixed attribute value
    
    # Metadata
    is_required: bool = False
    original_type_name: Optional[str] = None    # Original XSD type name for debugging
    
    # Diagnostic tracking
    extraction_failed: bool = False             # True if constraint extraction had errors
    default_fallback: bool = False              # True if using default constraints

    def effective_min(self) -> Optional[float]:
        """Get effective minimum considering both inclusive and exclusive bounds."""
        if self.min_value is not None and self.min_exclusive is not None:
            return max(self.min_value, self.min_exclusive + 0.0001)
        return self.min_value if self.min_value is not None else self.min_exclusive
    
    def effective_max(self) -> Optional[float]:
        """Get effective maximum considering both inclusive and exclusive bounds."""
        if self.max_value is not None and self.max_exclusive is not None:
            return min(self.max_value, self.max_exclusive - 0.0001)
        return self.max_value if self.max_value is not None else self.max_exclusive


# =============================================================================
# SchemaField: Represents a field in the schema with full metadata
# =============================================================================
@dataclass
class SchemaField:
    """
    Represents a single field in the Spark schema with all associated metadata.
    """
    name: str                                   # Field name (possibly flattened)
    spark_type: DataType
    nullable: bool
    constraints: TypeConstraints
    path: list[str] = field(default_factory=list)  # Path for nested fields
    
    @property
    def full_path(self) -> str:
        """Returns the full path as a string (e.g., 'AddressGrp.City')."""
        return ".".join(self.path + [self.name]) if self.path else self.name


# =============================================================================
# FieldHint: Semantic hints for realistic data generation
# =============================================================================
class FieldHint(Enum):
    """
    Semantic hints for REALISTIC text generation via Faker.
    Only used for free-form text fields where XSD just defines length/pattern.
    XSD enumerations, numeric patterns, etc. are handled directly by the XSD.
    """
    # Contact/Location - Faker makes these readable
    EMAIL = "email"
    CITY = "city"
    ADDRESS = "address"
    URL = "url"
    
    # Names - XSD has pattern but Faker gives real names
    PERSON_NAME = "person_name"
    BUSINESS_NAME = "business_name"
    
    # Text content - Faker generates readable sentences/paragraphs
    DESCRIPTION = "description"                 # Short description (sentence)
    EXPLANATION = "explanation"                 # Longer explanation (paragraph)
    TEXT = "text"                              # General text content
    


# Default patterns for auto-detecting field hints from names
# IMPROVED: Better text field detection
DEFAULT_FIELD_HINT_PATTERNS: dict[str, FieldHint] = {
    # Contact - Faker generates realistic values
    r'.*Email.*': FieldHint.EMAIL,
    r'.*Website.*|.*URL.*': FieldHint.URL,
    
    # Location - XSD has pattern but Faker makes it readable
    r'.*City(Nm|Name)$|^City$': FieldHint.CITY,
    r'.*Address.*Line.*|.*Street.*': FieldHint.ADDRESS,
    
    # Names - XSD has pattern but Faker gives real names
    r'.*Person.*Nm|.*Officer.*Nm|.*Individual.*Nm': FieldHint.PERSON_NAME,
    r'.*Business.*(Nm|Name)|.*Organization.*(Nm|Name)|.*Company.*(Nm|Name)': FieldHint.BUSINESS_NAME,
    
    # Text content - Faker generates readable sentences
    r'.*Desc(ription)?.*': FieldHint.DESCRIPTION,           # ActivityDesc, SpecialConditionDesc
    r'.*Explanation.*': FieldHint.EXPLANATION,              # LineExplanationType, etc.
    r'.*Detail.*|.*Comment.*|.*Note.*': FieldHint.TEXT,    # DetailTxt, CommentTxt, NoteTxt
    r'.*(Txt|Text)$': FieldHint.TEXT,                      # Generic text fields (end-anchored)
}


# =============================================================================
# FlattenMode: How to handle complex nested types
# =============================================================================
class FlattenMode(Enum):
    """
    Determines how complex/nested XSD types are converted to Spark schema.
    """
    NONE = "none"       # Preserve nested StructTypes
    FULL = "full"       # Flatten all nested types with prefixed names
    DEPTH = "depth"     # Flatten up to a specified depth


print("Core data structures defined: TypeConstraints (with min/max_occurs), SchemaField, FieldHint (with text hints), FlattenMode")

## TypeMapper

In [ ]:
class TypeMapper:
    """Maps XSD types to Spark SQL types via recursive base_type resolution."""

    # Add logger instance
    _logger = logging.getLogger('xsd_synthetic.TypeMapper')

    # Python types to Spark types mapping
    PYTHON_TO_SPARK = {
        int: LongType(),
        float: DoubleType(),
        str: StringType(),
        bool: BooleanType(),
        bytes: StringType(),
        type(None): StringType(),
    }

    # Import Decimal for decimal type mapping
    try:
        from decimal import Decimal as DecimalPy
        PYTHON_TO_SPARK[DecimalPy] = DecimalType(38, 18)
    except ImportError:
        pass

    # Import datetime types
    try:
        from datetime import date, datetime, time
        PYTHON_TO_SPARK[date] = DateType()
        PYTHON_TO_SPARK[datetime] = TimestampType()
        PYTHON_TO_SPARK[time] = StringType()
    except ImportError:
        pass

    # Import elementpath datetime types (used by xmlschema)
    try:
        from elementpath.datatypes.datetime import Date10, DateTime10, Time
        PYTHON_TO_SPARK[Date10] = DateType()
        PYTHON_TO_SPARK[DateTime10] = TimestampType()
        PYTHON_TO_SPARK[Time] = StringType()
    except ImportError:
        pass

    # NO SEMANTIC OVERRIDES - trust the XSD completely
    # Previous overrides removed because:
    # - CheckboxType: some are boolean, some are 'X' strings - trust XSD
    # - YearType: "2024" as string is valid, downstream can cast if needed
    # - MonthType: format is "--01" with dashes - CANNOT be integer!
    SEMANTIC_OVERRIDES = {}

    _type_cache: dict[str, Any] = {}

    @classmethod
    def clear_cache(cls):
        """Clear the type cache."""
        cls._type_cache = {}

    @staticmethod
    def get_spark_type(xsd_type, constraints: 'TypeConstraints' = None) -> Any:
        """Map XSD type to Spark type via recursive base_type resolution."""
        # Case 1: String type name (legacy)
        if isinstance(xsd_type, str):
            return TypeMapper._map_type_name(xsd_type, constraints)

        # Case 2: xmlschema type object
        if hasattr(xsd_type, 'name'):
            type_name = TypeMapper._clean_type_name(xsd_type.name)
            
            # Handle None type name (anonymous simpleType with restriction)
            if type_name is None or type_name == 'None':
                TypeMapper._logger.debug(
                    f"Anonymous type detected (name=None) - checking for base type"
                )
                
                if hasattr(xsd_type, 'base_type') and xsd_type.base_type:
                    base_type_name = TypeMapper._clean_type_name(
                        xsd_type.base_type.name if hasattr(xsd_type.base_type, 'name') else None
                    )
                    TypeMapper._logger.debug(
                        f"Anonymous type restricts base type: {base_type_name or 'unknown'} - recursively resolving"
                    )
                    return TypeMapper.get_spark_type(xsd_type.base_type, constraints)
                
                TypeMapper._logger.warning(
                    f"Anonymous type with no base_type - attempting to infer from python_type"
                )
                inferred_type = TypeMapper._infer_type_from_python_type(xsd_type)
                if inferred_type:
                    return TypeMapper._apply_constraints(inferred_type, constraints)
                
                TypeMapper._logger.warning(
                    f"Could not resolve anonymous type - falling back to StringType"
                )
                return StringType()

            # Check cache
            if type_name in TypeMapper._type_cache:
                cached_type = TypeMapper._type_cache[type_name]
                TypeMapper._logger.debug(f"Type '{type_name}' retrieved from cache: {cached_type}")
                return TypeMapper._apply_constraints(cached_type, constraints)

            # Layer 1: Semantic overrides (business decisions, not XSD fixes)
            if type_name in TypeMapper.SEMANTIC_OVERRIDES:
                spark_type = TypeMapper.SEMANTIC_OVERRIDES[type_name]
                TypeMapper._type_cache[type_name] = spark_type
                TypeMapper._logger.debug(f"Type '{type_name}' mapped to {spark_type} via semantic override")
                return spark_type

            # Layer 2: Recursive base_type resolution (proper XSD traversal)
            if hasattr(xsd_type, 'base_type') and xsd_type.base_type:
                spark_type = TypeMapper._resolve_base_type_recursively(xsd_type, type_name)
                if spark_type:
                    return TypeMapper._apply_constraints(spark_type, constraints)

            # Layer 3: Direct python_type (for unrestricted types)
            if hasattr(xsd_type, 'python_type'):
                python_type = xsd_type.python_type
                # Skip str - it's not informative, fall through to StringType
                if python_type != str:
                    spark_type = TypeMapper.PYTHON_TO_SPARK.get(python_type)
                    if spark_type:
                        TypeMapper._type_cache[type_name] = spark_type
                        TypeMapper._logger.debug(
                            f"Type '{type_name}' mapped to {spark_type} via python_type "
                            f"({python_type.__name__})"
                        )
                        return TypeMapper._apply_constraints(spark_type, constraints)

            # Fallback to StringType (no more name-based guessing)
            TypeMapper._logger.debug(
                f"Type '{type_name}' using StringType fallback"
            )

        return StringType()

    @staticmethod
    def _resolve_base_type_recursively(xsd_type, type_name: str, max_depth: int = 5) -> Any:
        """Walk base_type chain to find a mappable python_type."""
        current_type = xsd_type
        depth = 0
        
        while depth < max_depth:
            if not hasattr(current_type, 'base_type') or not current_type.base_type:
                break
                
            base_type = current_type.base_type
            
            # Try to map the base_type's python_type
            if hasattr(base_type, 'python_type'):
                base_python_type = base_type.python_type
                spark_type = TypeMapper.PYTHON_TO_SPARK.get(base_python_type)
                
                if spark_type:
                    # Check if python_type is str (ambiguous - keep looking)
                    if base_python_type == str and depth < max_depth - 1:
                        TypeMapper._logger.debug(
                            f"Type '{type_name}' base_type level {depth + 1} has python_type=str, "
                            f"continuing to search deeper"
                        )
                        current_type = base_type
                        depth += 1
                        continue
                    
                    # Found a good mapping!
                    TypeMapper._type_cache[type_name] = spark_type
                    TypeMapper._logger.debug(
                        f"Type '{type_name}' mapped to {spark_type} via Layer 2 "
                        f"(base_type chain depth={depth + 1}, python_type={base_python_type.__name__})"
                    )
                    return spark_type
            
            # Move to next level in hierarchy
            current_type = base_type
            depth += 1
        
        return None

    @staticmethod
    def _infer_type_from_python_type(xsd_type) -> Any:
        """Infer Spark type from python_type attribute."""
        if hasattr(xsd_type, 'python_type'):
            python_type = xsd_type.python_type
            spark_type = TypeMapper.PYTHON_TO_SPARK.get(python_type)
            if spark_type:
                TypeMapper._logger.debug(
                    f"Inferred type {spark_type} from python_type={python_type.__name__}"
                )
                return spark_type
        
        return None

    # XSD built-in primitive and derived types
    # These are the types that NumericBoundsStrategy checks against
    XSD_PRIMITIVE_TYPES = {
        # String types
        'string', 'normalizedString', 'token', 'language', 'Name', 'NCName',
        'NMTOKEN', 'NMTOKENS', 'ID', 'IDREF', 'IDREFS', 'ENTITY', 'ENTITIES',
        # Numeric types
        'decimal', 'integer', 'int', 'long', 'short', 'byte',
        'nonPositiveInteger', 'negativeInteger', 'nonNegativeInteger', 
        'positiveInteger', 'unsignedLong', 'unsignedInt', 'unsignedShort', 'unsignedByte',
        'float', 'double',
        # Date/time types
        'date', 'dateTime', 'time', 'gYear', 'gYearMonth', 'gMonth', 'gMonthDay', 'gDay',
        'duration',
        # Boolean
        'boolean',
        # Binary
        'hexBinary', 'base64Binary',
        # Other
        'anyURI', 'QName', 'NOTATION', 'anyType', 'anySimpleType',
    }

    @staticmethod
    def get_base_xsd_type(xsd_type, max_depth: int = 10) -> str:
        """Get primitive XSD base type by traversing type hierarchy."""
        if isinstance(xsd_type, str):
            return TypeMapper._clean_type_name(xsd_type)
        
        depth = 0
        current_type = xsd_type
        
        while depth < max_depth:
            if current_type is None:
                break
                
            # Get the type name
            type_name = None
            if hasattr(current_type, 'name') and current_type.name:
                type_name = TypeMapper._clean_type_name(current_type.name)
            
            # If we have a primitive type name, we're done
            if type_name and type_name in TypeMapper.XSD_PRIMITIVE_TYPES:
                TypeMapper._logger.debug(
                    f"get_base_xsd_type resolved to primitive '{type_name}' at depth {depth}"
                )
                return type_name
            
            # Try to go deeper via base_type
            if hasattr(current_type, 'base_type') and current_type.base_type:
                current_type = current_type.base_type
                depth += 1
                continue
            
            # For XsdComplexType with simpleContent, check content attribute
            if hasattr(current_type, 'content') and current_type.content:
                content = current_type.content
                # If content is also a type, recurse into it
                if hasattr(content, 'base_type') and content.base_type:
                    current_type = content.base_type
                    depth += 1
                    continue
            
            # No more parents to traverse - return what we have
            if type_name:
                TypeMapper._logger.debug(
                    f"get_base_xsd_type stopped at '{type_name}' (not primitive, depth {depth})"
                )
                return type_name
            
            break
        
        TypeMapper._logger.debug(
            f"get_base_xsd_type defaulting to 'string' after {depth} iterations"
        )
        return 'string'

    @staticmethod
    def _get_local_name(xsd_type) -> str:
        """Get the local name of an XSD type."""
        if hasattr(xsd_type, 'name'):
            cleaned = TypeMapper._clean_type_name(xsd_type.name)
            # Return None if name is None (don't convert to string 'None')
            if cleaned is None:
                return None
            return cleaned
        return str(xsd_type)

    @staticmethod
    def _map_type_name(type_name: str, constraints: 'TypeConstraints' = None) -> Any:
        """Map a type name string to Spark type (legacy string-based lookup)."""
        type_name = TypeMapper._clean_type_name(type_name)
        
        # Handle None or 'None' type name
        if type_name is None or type_name == 'None':
            return StringType()

        # Check semantic overrides
        if type_name in TypeMapper.SEMANTIC_OVERRIDES:
            return TypeMapper.SEMANTIC_OVERRIDES[type_name]

        # XSD primitive type mappings
        basic_map = {
            'string': StringType(),
            'integer': LongType(),
            'int': IntegerType(),
            'long': LongType(),
            'short': ShortType(),
            'byte': ByteType(),
            'decimal': DecimalType(38, 18),
            'float': FloatType(),
            'double': DoubleType(),
            'boolean': BooleanType(),
            'date': DateType(),
            'dateTime': TimestampType(),
            'time': StringType(),
        }

        spark_type = basic_map.get(type_name, StringType())
        return TypeMapper._apply_constraints(spark_type, constraints)

    @staticmethod
    def _clean_type_name(name: str) -> str:
        """Remove namespace prefix from type name."""
        if not name:
            return name
        if '}' in name:
            return name.split('}')[1]
        return name

    @staticmethod
    def _apply_constraints(spark_type: Any, constraints: 'TypeConstraints') -> Any:
        """Apply constraints to Spark type."""
        if not constraints:
            return spark_type

        if isinstance(spark_type, DecimalType):
            if constraints.total_digits:
                precision = min(constraints.total_digits, 38)
                scale = min(constraints.fraction_digits or 0, precision)
                return DecimalType(precision, scale)

        return spark_type

    @classmethod
    def log_mapping_summary(cls):
        """Log summary of type mappings (call at end of schema build)."""
        type_distribution = {}
        for type_name, spark_type in cls._type_cache.items():
            spark_type_name = type(spark_type).__name__
            type_distribution[spark_type_name] = type_distribution.get(spark_type_name, 0) + 1
        
        cls._logger.info(f"Type mapping summary: {len(cls._type_cache)} unique types cached")
        for spark_type_name, count in sorted(type_distribution.items(), key=lambda x: -x[1]):
            cls._logger.info(f"  {spark_type_name}: {count} types")


print("TypeMapper class defined (v2.4: Recursive primitive type resolution for XsdComplexType)")

## ConstraintExtractor

In [ ]:
class ConstraintExtractor:
    """Extracts XSD facet restrictions, handling type inheritance."""
    
    # Add logger instance
    _logger = logging.getLogger('xsd_synthetic.ConstraintExtractor')
    
    @classmethod
    def extract(cls, xsd_type, type_mapper: TypeMapper = None) -> TypeConstraints:
        """
        Extract all constraints from an XSD type.
        
        Args:
            xsd_type: An xmlschema type object
            type_mapper: Optional TypeMapper instance (uses class methods if None)
            
        Returns:
            TypeConstraints with all extracted facet restrictions
        """
        if type_mapper is None:
            type_mapper = TypeMapper
        
        # Handle None or invalid types gracefully
        if xsd_type is None:
            constraints = TypeConstraints(base_type='string', spark_type=StringType())
            constraints.default_fallback = True
            cls._logger.warning("Constraint extraction received None type, using default StringType fallback")
            return constraints
        
        # Get base type and Spark type
        base_type = type_mapper.get_base_xsd_type(xsd_type)
        spark_type = type_mapper.get_spark_type(xsd_type)
        
        # Initialize constraints
        constraints = TypeConstraints(
            base_type=base_type,
            spark_type=spark_type,
            original_type_name=type_mapper._get_local_name(xsd_type)
        )
        
        # Extract facets by walking the type hierarchy
        try:
            cls._extract_facets_recursive(xsd_type, constraints)
        except Exception as e:
            # Mark that extraction failed
            constraints.extraction_failed = True
            cls._logger.error(
                f"Constraint extraction failed for type '{constraints.original_type_name}': "
                f"{type(e).__name__}: {e}"
            )
            # Still return constraints with base type info
        
        # Adjust Spark type based on extracted constraints
        constraints.spark_type = cls._refine_spark_type(constraints)
        
        # Store original XSD type for sample generation
        constraints.xsd_type = xsd_type
        
        return constraints
    
    @classmethod
    def _extract_facets_recursive(cls, xsd_type, constraints: TypeConstraints):
        """Recursively extract facets from type and base types."""
        if xsd_type is None:
            return
        
        type_name = type(xsd_type).__name__ if hasattr(xsd_type, '__class__') else str(xsd_type)
        cls._logger.debug(f"Traversing type hierarchy: {type_name}")
        
        # First, extract from base type (so child can override)
        if hasattr(xsd_type, 'base_type') and xsd_type.base_type is not None:
            try:
                cls._extract_facets_recursive(xsd_type.base_type, constraints)
            except Exception as e:
                cls._logger.debug(f"Failed to extract from base type: {e}")
                pass
        
        # Handle XsdList types - extract constraints from item_type
        # e.g., IdListType is a list of IdType, and IdType has the pattern
        if hasattr(xsd_type, 'item_type') and xsd_type.item_type is not None:
            cls._logger.debug(f"XsdList detected - extracting from item_type: {xsd_type.item_type}")
            try:
                cls._extract_facets_recursive(xsd_type.item_type, constraints)
            except Exception as e:
                cls._logger.debug(f"Failed to extract from item_type: {e}")
                pass
        
        # Handle XsdUnion types - extract constraints from all member_types
        # e.g., AllCountriesType is a union of 'US' enum and CountryType enum
        # Merge enumerations from all members since value can match any
        if hasattr(xsd_type, 'member_types') and xsd_type.member_types:
            cls._logger.debug(f"XsdUnion detected - extracting from {len(xsd_type.member_types)} member_types")
            for member in xsd_type.member_types:
                try:
                    cls._extract_facets_recursive(member, constraints)
                except Exception as e:
                    cls._logger.debug(f"Failed to extract from member_type: {e}")
                    pass
        
        # Handle XsdComplexType with simpleContent - extract from simple_type or content
        # e.g., GrossReceiptsOrSalesAmt is complexType extending USAmountType with attributes
        # IMPORTANT: Only do this for XsdComplexType, not XsdAtomicBuiltin (which has self-referential simple_type)
        type_class = type(xsd_type).__name__
        if type_class == 'XsdComplexType':
            if hasattr(xsd_type, 'simple_type') and xsd_type.simple_type is not None:
                cls._logger.debug(f"XsdComplexType with simpleContent - extracting from simple_type: {xsd_type.simple_type}")
                try:
                    cls._extract_facets_recursive(xsd_type.simple_type, constraints)
                except Exception as e:
                    cls._logger.debug(f"Failed to extract from simple_type: {e}")
                    pass
            elif hasattr(xsd_type, 'content') and xsd_type.content is not None:
                # Fallback to content if simple_type not available
                content = xsd_type.content
                # Only follow if content is a simple type (not XsdGroup)
                content_class = type(content).__name__
                if 'Atomic' in content_class or 'Restriction' in content_class:
                    cls._logger.debug(f"XsdComplexType - extracting from content: {content}")
                    try:
                        cls._extract_facets_recursive(content, constraints)
                    except Exception as e:
                        cls._logger.debug(f"Failed to extract from content: {e}")
                        pass
        
        # Now extract facets from this type
        cls._extract_facets(xsd_type, constraints)
    
    @classmethod
    def _extract_facets(cls, xsd_type, constraints: TypeConstraints):
        """Extract facets from a single type level."""
        
        # Skip if not a proper xmlschema type object
        if not hasattr(xsd_type, '__class__') or 'xmlschema' not in str(type(xsd_type).__module__):
            return
        
        # Try different ways to access facets based on xmlschema version
        facets = None
        
        try:
            if hasattr(xsd_type, 'facets'):
                facets = xsd_type.facets
            elif hasattr(xsd_type, 'validators'):
                facets = {type(v).__name__: v for v in xsd_type.validators if hasattr(v, 'value')}
        except:
            pass
        
        if facets:
            cls._process_facets(facets, constraints)
        
        # Also check for enumeration directly on type
        try:
            if hasattr(xsd_type, 'enumeration') and xsd_type.enumeration:
                constraints.enumeration = list(xsd_type.enumeration)
        except:
            pass
        
        # Check for pattern (FIXED: XsdPatternFacets handling)
        try:
            if hasattr(xsd_type, 'patterns') and xsd_type.patterns:
                # FIXED: XsdPatternFacets has a .regexps attribute with pattern strings
                if hasattr(xsd_type.patterns, 'regexps') and xsd_type.patterns.regexps:
                    # Use the first pattern (most common case)
                    constraints.pattern = xsd_type.patterns.regexps[0]
                    cls._logger.debug(f"Extracted pattern: {constraints.pattern}")
                else:
                    # Fallback: iterate and get 'value' attribute from ElementTree.Element
                    for pattern_elem in xsd_type.patterns:
                        if hasattr(pattern_elem, 'get'):
                            pattern_value = pattern_elem.get('value')
                            if pattern_value:
                                constraints.pattern = pattern_value
                                cls._logger.debug(f"Extracted pattern from element: {constraints.pattern}")
                                break
        except Exception as e:
            cls._logger.debug(f"Pattern extraction failed: {e}")
            pass
    
    
    # Declarative facet mapping configuration
    from dataclasses import dataclass
    from typing import Callable, Any

    @dataclass
    class FacetHandler:
        """Configuration for handling a specific facet type."""
        keywords: list[str]  # Keywords to match in facet_key
        attribute: str  # Constraint attribute to set
        converter: Callable[[Any], Any]  # Value conversion function
        exact_match: bool = False  # If True, require exact match

    # Facet lookup table (declarative configuration)
    FACET_HANDLERS = [
        # Numeric bounds
        # FIXED: Use 'v is not None' instead of 'if v' to handle 0 values correctly
        FacetHandler(['mininclusive'], 'min_value', lambda v: float(v) if v is not None else None),
        FacetHandler(['maxinclusive'], 'max_value', lambda v: float(v) if v is not None else None),
        FacetHandler(['minexclusive'], 'min_exclusive', lambda v: float(v) if v is not None else None),
        FacetHandler(['maxexclusive'], 'max_exclusive', lambda v: float(v) if v is not None else None),

        # String length
        FacetHandler(['minlength'], 'min_length', int),
        FacetHandler(['maxlength'], 'max_length', int),
        FacetHandler(['length'], 'length', int, exact_match=True),

        # Numeric precision
        FacetHandler(['totaldigits'], 'total_digits', int),
        FacetHandler(['fractiondigits'], 'fraction_digits', int),

        # Pattern
        FacetHandler(['pattern'], 'pattern', str),

        # Enumeration (special handling)
        FacetHandler(['enumeration'], 'enumeration', None),
    ]

    @classmethod
    def _find_facet_handler(cls, facet_key: str):
        """Find the handler for a facet key."""
        for handler in cls.FACET_HANDLERS:
            for keyword in handler.keywords:
                if handler.exact_match:
                    if facet_key == keyword:
                        return handler
                else:
                    if keyword in facet_key:
                        return handler
        return None

    @classmethod
    def _apply_facet_handler(
        cls,
        handler,
        constraints: TypeConstraints,
        value: Any,
        facet_key: str,
        facets_found: list
    ):
        """Apply a facet handler to constraints."""
        try:
            # Special case: enumeration
            if handler.attribute == 'enumeration':
                if constraints.enumeration is None:
                    constraints.enumeration = []
                if isinstance(value, (list, tuple)):
                    constraints.enumeration.extend(value)
                else:
                    constraints.enumeration.append(str(value))
                facets_found.append("enumeration")
                return

            # General case: convert and set attribute
            if handler.converter:
                converted_value = handler.converter(value)
                setattr(constraints, handler.attribute, converted_value)
                facets_found.append(f"{facet_key}={value}")
        except Exception as e:
            cls._logger.debug(f"Failed to process facet {facet_key}: {e}")

    @classmethod
    def _is_valid_facet(cls, facet) -> bool:
        """Check if facet object is valid for processing."""
        if facet is None:
            return False

        # Skip ElementTree.Element objects
        facet_type = type(facet).__module__
        if 'xml.etree' in facet_type or 'ElementTree' in str(type(facet)):
            return False

        # Must have value attribute
        if not hasattr(facet, 'value'):
            return False

        return True

    
    @classmethod
    def _process_facets(cls, facets: dict, constraints: TypeConstraints):
        """Process XSD facets using FACET_HANDLERS lookup table."""
        facets_found = []

        for facet_name, facet in facets.items():
            # Validation
            if not cls._is_valid_facet(facet):
                continue

            value = facet.value
            if value is None:
                continue

            facet_key = facet_name.lower().replace('xsd', '').replace('facet', '')

            # Find and apply matching handler
            handler = cls._find_facet_handler(facet_key)
            if handler:
                cls._apply_facet_handler(handler, constraints, value, facet_key, facets_found)

        if facets_found:
            cls._logger.debug(f"Extracted facets: {', '.join(facets_found)}")
    @classmethod
    def _to_number(cls, value) -> Optional[float]:
        """Safely convert a value to a number."""
        try:
            return float(value)
        except (TypeError, ValueError):
            return None
    
    @classmethod
    def _refine_spark_type(cls, constraints: TypeConstraints) -> DataType:
        """Refine Spark type based on constraints (e.g., DecimalType precision)."""
        spark_type = constraints.spark_type
        
        # Refine decimal types based on totalDigits/fractionDigits facets
        # Works for any type that resolved to DecimalType (USDecimalAmountType, DecimalType, etc.)
        if isinstance(spark_type, DecimalType) and constraints.total_digits:
            total = min(constraints.total_digits, 38)
            frac = min(constraints.fraction_digits or 0, total)
            return DecimalType(total, frac)
        
        # Use IntegerType for small integers, LongType for large
        base = constraints.base_type
        if base in ('integer', 'nonNegativeInteger', 'positiveInteger', 'nonPositiveInteger', 'negativeInteger'):
            if constraints.total_digits and constraints.total_digits <= 9:
                return IntegerType()
            return LongType()
        
        return spark_type


print("ConstraintExtractor class defined")

## SparkSchemaBuilder

In [ ]:
class SparkSchemaBuilder:
    """Builds Spark StructType from XSD schema with configurable flattening."""
    
    # Add logger instance
    _logger = logging.getLogger('xsd_synthetic.SparkSchemaBuilder')
    
    def __init__(
        self, 
        flatten_mode: FlattenMode = FlattenMode.NONE,
        max_depth: int = 10,
        separator: str = "_",
        exclude_root_from_path: bool = True
    ):
        """
        Initialize the schema builder.
        
        Args:
            flatten_mode: How to handle nested complex types
            max_depth: Maximum depth for DEPTH mode or recursion limit
            separator: Separator for flattened field names (e.g., "AddressGrp_City")
            exclude_root_from_path: Whether to exclude root element name from flattened paths (default: True)
        """
        self.flatten_mode = flatten_mode
        self.max_depth = max_depth
        self.separator = separator
        self.exclude_root_from_path = exclude_root_from_path
        self.type_mapper = TypeMapper
        self.constraint_extractor = ConstraintExtractor
    
    def build(
        self, 
        schema: 'xmlschema.XMLSchema', 
        root_element: str
    ) -> tuple[StructType, dict[str, TypeConstraints]]:
        """
        Build a Spark schema from an XSD schema.
        
        Args:
            schema: Loaded xmlschema.XMLSchema object
            root_element: Name of the root element to process
            
        Returns:
            Tuple of (StructType, dict mapping field names to TypeConstraints)
        """
        self._logger.info(f"Building Spark schema for root element: '{root_element}'")
        self._logger.debug(f"Flatten mode: {self.flatten_mode.value}, max_depth: {self.max_depth}")
        
        # IMPROVED: Clear type cache to avoid cross-schema pollution
        self.type_mapper.clear_cache()
        
        # Find the root element
        element = schema.elements.get(root_element)
        if element is None:
            raise ValueError(f"Root element '{root_element}' not found in schema. "
                           f"Available elements: {list(schema.elements.keys())}")
        
        # Process the element's type
        fields: list[SchemaField] = []
        self._process_element(element, fields, path=[], depth=0)
        
        # Build StructType and constraints dict
        struct_fields = []
        constraints_map = {}
        
        for field in fields:
            struct_fields.append(StructField(field.name, field.spark_type, field.nullable))
            constraints_map[field.name] = field.constraints
        
        # Count extraction issues
        failed_count = sum(1 for c in constraints_map.values() if c.extraction_failed)
        fallback_count = sum(1 for c in constraints_map.values() if c.default_fallback)
        
        self._logger.info(
            f"Schema build complete: {len(struct_fields)} fields generated "
            f"({failed_count} extraction failures, {fallback_count} default fallbacks)"
        )
        
        # Call TypeMapper summary
        self.type_mapper.log_mapping_summary()
        
        return StructType(struct_fields), constraints_map
    
    def _process_element(
        self, 
        element, 
        fields: list[SchemaField], 
        path: list[str], 
        depth: int
    ):
        """Process an XSD element and its children recursively."""
        
        element_name = self._get_local_name(element.name)
        full_path = '.'.join(path + [element_name]) if path else element_name
        
        self._logger.debug(f"Processing element: {full_path} (depth={depth})")
        
        if depth > self.max_depth:
            self._logger.warning(f"Max depth {self.max_depth} exceeded at path '{full_path}'")
            warnings.warn(f"Max depth {self.max_depth} exceeded at path {'.'.join(path)}")
            return
        
        # IMPROVED: Add error handling for malformed schemas
        try:
            element_type = element.type
            
            # Check if it's a complex type with child elements
            if self._is_complex_type(element_type):
                self._process_complex_type(element, element_type, fields, path, depth)
            else:
                # Simple type - add as a field
                self._add_simple_field(element, fields, path)
        
        except Exception as e:
            # Only warn if it's not an ElementTree.Element issue
            field_path = '.'.join(path + [element_name]) if path else element_name
            
            if 'ElementTree' not in str(e):
                self._logger.error(
                    f"Failed to process element '{field_path}': {type(e).__name__}: {e}"
                )
                warnings.warn(f"Error processing element '{field_path}': {e}. Adding as string field.")
            else:
                self._logger.debug(
                    f"Skipping ElementTree.Element at '{field_path}' (expected behavior)"
                )
            
            # Add as string field fallback (silently for ElementTree issues)
            field_name = self._build_field_name(element_name, path)
            # Use case-insensitive comparison for Spark SQL compatibility
            if not any(f.name.lower() == field_name.lower() for f in fields):
                constraints = TypeConstraints(base_type='string', spark_type=StringType())
                constraints.extraction_failed = True
                fields.append(SchemaField(
                    name=field_name,
                    spark_type=StringType(),
                    nullable=True,
                    constraints=constraints,
                    path=path
                ))
    
    def _is_complex_type(self, xsd_type) -> bool:
        """Check if an XSD type is complex (has child elements)."""
        if xsd_type is None:
            return False
        
        # Check for content model with child elements
        if hasattr(xsd_type, 'content'):
            content = xsd_type.content
            if hasattr(content, 'iter_elements'):
                try:
                    elements = list(content.iter_elements())
                    return len(elements) > 0
                except:
                    pass
        
        return hasattr(xsd_type, 'content_type') and xsd_type.content_type == 'element-only'
    
    def _process_complex_type(
        self, 
        element, 
        element_type, 
        fields: list[SchemaField], 
        path: list[str], 
        depth: int
    ):
        """Process a complex type with child elements."""
        
        element_name = self._get_local_name(element.name)
        new_path = path + [element_name]
        
        # Get child elements
        child_elements = []
        if hasattr(element_type, 'content') and hasattr(element_type.content, 'iter_elements'):
            try:
                child_elements = list(element_type.content.iter_elements())
            except Exception as e:
                self._logger.debug(f"Failed to iterate child elements for '{element_name}': {e}")
                pass
        
        if not child_elements:
            self._logger.debug(f"Complex type '{element_name}' has no children, treating as simple type")
            # No children found - treat as simple type
            self._add_simple_field(element, fields, path)
            return
        
        # Decide based on flatten mode
        should_flatten = self._should_flatten(depth)
        
        self._logger.debug(
            f"Processing complex type '{element_name}' with {len(child_elements)} children "
            f"(flatten={should_flatten})"
        )
        
        if should_flatten:
            # Add attributes of this complex type FIRST (so root element attributes take precedence)
            self._add_attributes(element, fields, new_path if not self.exclude_root_from_path or depth > 0 else [])
            
            # Then flatten: process children with extended path
            for child in child_elements:
                self._process_element(child, fields, new_path, depth + 1)
        else:
            # Nested: create a StructType for this complex element
            nested_fields: list[SchemaField] = []
            for child in child_elements:
                self._process_element(child, nested_fields, [], depth + 1)
            
            if nested_fields:
                # Create nested StructType
                nested_struct = StructType([
                    StructField(f.name, f.spark_type, f.nullable) 
                    for f in nested_fields
                ])
                
                # Add the complex field
                field_name = self._build_field_name(element_name, path)
                constraints = TypeConstraints(
                    base_type='complex',
                    spark_type=nested_struct,
                    is_required=element.min_occurs > 0,
                    original_type_name=self._get_local_name(element_type)
                )
                
                # Check if complex type is an array
                spark_type = nested_struct
                is_array = hasattr(element, 'max_occurs') and (element.max_occurs is None or element.max_occurs > 1)

                if is_array:
                    spark_type = ArrayType(nested_struct)

                # Add attributes to the complex type's nested fields
                self._add_attributes(element, nested_fields, [])

                # Recreate nested struct with attributes included
                if nested_fields:
                    nested_struct = StructType([
                        StructField(f.name, f.spark_type, f.nullable)
                        for f in nested_fields
                    ])
                    if is_array:
                        spark_type = ArrayType(nested_struct)
                    else:
                        spark_type = nested_struct

                fields.append(SchemaField(
                    name=field_name,
                    spark_type=spark_type,
                    nullable=element.min_occurs == 0,
                    constraints=constraints,
                    path=path
                ))
    
    def _add_simple_field(
        self,
        element,
        fields: list[SchemaField],
        path: list[str]
    ):
        """Add a simple (non-complex) field to the fields list."""

        element_name = self._get_local_name(element.name)
        field_name = self._build_field_name(element_name, path)

        # Handle duplicate field names by adding suffix counter
        # This occurs with XSD <choice> elements that have same-named elements in different branches
        # Use case-insensitive comparison because Spark SQL is case-insensitive by default
        base_name = field_name
        counter = 1
        while any(f.name.lower() == field_name.lower() for f in fields):
            field_name = f"{base_name}_{counter}"
            counter += 1
            self._logger.debug(f"Renamed duplicate field '{base_name}' to '{field_name}' (case-insensitive collision)")

        # Check if element.type is valid before extracting constraints
        element_type = element.type

        # Skip ElementTree.Element objects - create basic string constraint
        if element_type is not None:
            type_module = str(type(element_type).__module__)
            if 'xml.etree' in type_module or 'ElementTree' in str(type(element_type)):
                # ElementTree.Element - use basic string constraint
                constraints = TypeConstraints(base_type='string', spark_type=StringType())
                constraints.extraction_failed = True
            else:
                # Valid xmlschema type - extract constraints normally
                constraints = self.constraint_extractor.extract(element_type)
        else:
            # No type info - use basic string constraint
            constraints = TypeConstraints(base_type='string', spark_type=StringType())
            constraints.default_fallback = True

        # Track occurrence constraints from element (for arrays)
        # NOTE: min_occurs/max_occurs are set for documentation, but synthetic
        # data generation cannot currently enforce array element counts
        if hasattr(element, 'min_occurs'):
            constraints.min_occurs = element.min_occurs
        if hasattr(element, 'max_occurs'):
            constraints.max_occurs = element.max_occurs

        # Set is_required based on minOccurs
        constraints.is_required = element.min_occurs > 0 if hasattr(element, 'min_occurs') else False

        # Check if this is an array (repeated element)
        spark_type = constraints.spark_type
        is_array = hasattr(element, 'max_occurs') and (element.max_occurs is None or element.max_occurs > 1)

        if is_array:
            # Wrap in ArrayType for repeated elements
            spark_type = ArrayType(spark_type)

            # Warn about array element count constraints that can't be enforced
            # TODO: Implement custom array generation with UDFs to respect minOccurs/maxOccurs
            if constraints.min_occurs and constraints.min_occurs > 0:
                self._logger.warning(
                    f"Array field '{field_name}' has minOccurs={constraints.min_occurs}, "
                    f"but synthetic data generation cannot enforce array occurrence constraints. "
                    f"The field will be nullable={element.min_occurs == 0}, but array element counts are not controlled. "
                    f"TODO: Implement custom array generation."
                )

        fields.append(SchemaField(
            name=field_name,
            spark_type=spark_type,
            nullable=element.min_occurs == 0,
            constraints=constraints,
            path=path
        ))

        # Add attributes if this element has any
        self._add_attributes(element, fields, path)
    def _add_attributes(
        self,
        element,
        fields: list[SchemaField],
        path: list[str]
    ):
        """Add XML attributes as fields in the Spark schema."""

        if not hasattr(element, 'type') or element.type is None:
            return

        elem_type = element.type

        # Check if type has attributes
        if not hasattr(elem_type, 'attributes'):
            return

        attrs = elem_type.attributes
        if not attrs:
            return

        # Iterate through attributes
        try:
            if hasattr(attrs, 'items'):
                # XsdAttributeGroup (dict-like)
                for attr_name, attr in attrs.items():
                    self._add_attribute_field(attr, attr_name, fields, path)
            elif hasattr(attrs, '__iter__'):
                # Iterable
                for attr in attrs:
                    attr_name = attr.name if hasattr(attr, 'name') else str(attr)
                    self._add_attribute_field(attr, attr_name, fields, path)
        except Exception as e:
            # Silently skip if attribute iteration fails
            pass

    def _add_attribute_field(
        self,
        attr,
        attr_name: str,
        fields: list[SchemaField],
        path: list[str]
    ):
        """Add a single attribute as a field."""

        # Clean attribute name
        attr_name = self._get_local_name(attr_name)

        # Build field name (attributes use same naming as elements)
        field_name = self._build_field_name(attr_name, path)

        # Skip if already exists (case-insensitive for Spark SQL compatibility)
        if any(f.name.lower() == field_name.lower() for f in fields):
            return

        # Get attribute type
        attr_type = None
        if hasattr(attr, 'type'):
            attr_type = attr.type

        # Extract constraints
        if attr_type is not None:
            constraints = self.constraint_extractor.extract(attr_type)
        else:
            # No type info - default to string
            constraints = TypeConstraints(base_type='string', spark_type=StringType())
            constraints.default_fallback = True

        # Check if required
        is_required = False
        if hasattr(attr, 'use'):
            is_required = attr.use == 'required'

        constraints.is_required = is_required
        
        # Check for fixed value (XSD fixed attribute)
        if hasattr(attr, 'fixed') and attr.fixed is not None:
            constraints.fixed_value = str(attr.fixed)
            self._logger.debug(f"Attribute '{field_name}' has fixed value: '{attr.fixed}'")

        # Add field (attributes are nullable unless required)
        fields.append(SchemaField(
            name=field_name,
            spark_type=constraints.spark_type,
            nullable=not is_required,  # Required attrs are not nullable
            constraints=constraints,
            path=path
        ))

    def _should_flatten(self, depth: int) -> bool:
        """Determine if we should flatten at this depth."""
        if self.flatten_mode == FlattenMode.FULL:
            return True
        elif self.flatten_mode == FlattenMode.NONE:
            return False
        elif self.flatten_mode == FlattenMode.DEPTH:
            return depth < self.max_depth
        return False
    
    def _build_field_name(self, name: str, path: list[str]) -> str:
        """Build the field name, potentially with path prefix for flattened fields."""
        if path and self.flatten_mode != FlattenMode.NONE:
            # Optionally exclude root element from path
            effective_path = path[1:] if self.exclude_root_from_path and len(path) > 0 else path
            if effective_path:
                return self.separator.join(effective_path + [name])
        return name
    
    def _get_local_name(self, name) -> str:
        """Extract local name, stripping namespace."""
        name = str(name)
        if '}' in name:
            name = name.split('}')[1]
        return name

    def preview(self, schema: 'xmlschema.XMLSchema', root_element: str) -> None:
        """
        Preview schema without generating data - shows field count, types, potential issues.
        
        Use this to inspect an XSD before running full generation.
        """
        # Validate root element exists
        if root_element not in schema.elements:
            available = list(schema.elements.keys())[:10]
            print(f"[ERROR] Element '{root_element}' not found")
            print(f"  Available elements: {available}")
            return
        
        print(f"\n{'='*60}")
        print(f"Schema Preview: {root_element}")
        print(f"{'='*60}")
        
        # Build schema to get field info
        spark_schema, constraints = self.build(schema, root_element)
        
        # Type distribution
        type_counts = {}
        for field in spark_schema.fields:
            t = type(field.dataType).__name__
            type_counts[t] = type_counts.get(t, 0) + 1
        
        print(f"\nField Summary:")
        print(f"  Total fields: {len(spark_schema.fields)}")
        for t, count in sorted(type_counts.items(), key=lambda x: -x[1]):
            print(f"    {t}: {count}")
        
        # Check for issues
        issues = []
        for name, c in constraints.items():
            if c.extraction_failed:
                issues.append(f"  - {name}: constraint extraction failed")
            if isinstance(c.spark_type, StringType) and 'Amt' in name and not c.enumeration:
                issues.append(f"  - {name}: amount field mapped to string")
        
        if issues:
            print(f"\nPotential Issues ({len(issues)}):")
            for issue in issues[:10]:
                print(issue)
            if len(issues) > 10:
                print(f"  ... and {len(issues) - 10} more")
        else:
            print(f"\n[OK] No issues detected")
        
        # Sample fields
        print(f"\nSample Fields (first 10):")
        for field in spark_schema.fields[:10]:
            c = constraints.get(field.name)
            extras = []
            if c and c.enumeration:
                extras.append(f"enum[{len(c.enumeration)}]")
            if c and c.pattern:
                extras.append("pattern")
            extra_str = f" ({', '.join(extras)})" if extras else ""
            print(f"  {field.name}: {type(field.dataType).__name__}{extra_str}")
        
        print(f"{'='*60}\n")


print("SparkSchemaBuilder class defined (with preview mode)")

## SyntheticDataGenerator

In [ ]:
# =============================================================================
# Data Generation Helpers
# Simple, linear configuration - no complex strategy pattern
# =============================================================================

# Faker method mapping for realistic text generation
FAKER_METHODS = {
    'email': 'email',
    'city': 'city',
    'address': 'street_address',
    'street': 'street_address',
    'personn': 'name',        # PersonNm, OfficerNm
    'officern': 'name',
    'businessn': 'company',   # BusinessNm
    'companyn': 'company',
    'organizationn': 'company',
    'desc': 'sentence',       # ActivityDesc, etc.
    'explanation': 'paragraph',
}

def get_faker_method(field_name: str) -> str:
    """Get Faker method for realistic text, or None."""
    name_lower = field_name.lower()
    for pattern, method in FAKER_METHODS.items():
        if pattern in name_lower:
            return method
    return None

def get_dbldatagen_type(spark_type) -> str:
    """Convert Spark type to dbldatagen type string."""
    from pyspark.sql.types import (
        StringType, IntegerType, LongType, FloatType, DoubleType,
        DecimalType, BooleanType, DateType, TimestampType
    )
    match spark_type:
        case StringType():   return "string"
        case LongType():     return "long"
        case IntegerType():  return "int"
        case FloatType():    return "float"
        case DoubleType():   return "double"
        case DecimalType():  return f"decimal({spark_type.precision},{spark_type.scale})"
        case BooleanType():  return "boolean"
        case DateType():     return "date"
        case TimestampType(): return "timestamp"
        case _:              return "string"

def generate_pattern_samples(pattern: str, count: int = 100) -> list:
    """
    Generate sample strings matching an XSD regex pattern using rstr.
    
    Uses rstr.xeger() to generate valid strings from ANY regex pattern.
    No manual parsing needed - handles all XSD patterns automatically.
    """
    import rstr
    
    if not pattern:
        return []
    
    try:
        # Generate unique samples
        samples = set()
        for _ in range(count * 2):  # Generate extra to account for duplicates
            samples.add(rstr.xeger(pattern))
            if len(samples) >= count:
                break
        return list(samples)
    except Exception:
        return []  # Invalid pattern, fall back to other strategies

print("Data generation helpers defined (simplified - no strategy pattern)")


In [ ]:
class SyntheticDataGenerator:
    """Generates synthetic data using dbldatagen with Faker for realistic values."""
    
    # Add logger instance
    _logger = logging.getLogger('xsd_synthetic.SyntheticDataGenerator')
    
    
    # Faker provider mappings for realistic data generation
    # Only for free-form text fields - XSD handles patterns/enumerations
    FAKER_PROVIDERS = {
        FieldHint.PERSON_NAME: 'name',
        FieldHint.BUSINESS_NAME: 'company',
        FieldHint.CITY: 'city',
        FieldHint.ADDRESS: 'street_address',
        FieldHint.EMAIL: 'email',
        FieldHint.URL: 'url',
        # Text generation
        FieldHint.DESCRIPTION: 'sentence',     # Short description
        FieldHint.EXPLANATION: 'paragraph',    # Longer explanation
        FieldHint.TEXT: 'text',                # General text
    }
    
    def __init__(self, spark: SparkSession, schema=None, amount_ranges=None, date_range=None):
        """Initialize generator with SparkSession and optional settings."""
        self.spark = spark
        self.schema = schema  # NEW: For xmlschema sample generation
        self.amount_ranges = amount_ranges or {'default': (0, 1000000)}
        self.date_range = date_range or ('2020-01-01', '2025-12-31')
        self._import_dbldatagen()
        self._create_faker()
        # Simple configuration - no strategy pattern needed

    
    def _import_dbldatagen(self):
        """Import dbldatagen, with helpful error if not installed."""
        try:
            import dbldatagen as dg
            self.dg = dg
        except ImportError:
            raise ImportError(
                "dbldatagen is required for data generation. "
                "Install it with: pip install dbldatagen"
            )
    
    def _create_faker(self):
        """Create Faker factory for realistic data generation."""
        try:
            self.faker = self.dg.FakerTextFactory(locale=['en_US'])
            self._logger.info("Faker factory initialized successfully")
        except Exception as e:
            self._logger.warning(
                f"Faker initialization failed: {e}. "
                "Install faker with: pip install faker. "
                "Falling back to template-based generation."
            )
            warnings.warn(
                f"Faker initialization failed: {e}. "
                "Install faker with: pip install faker. "
                "Falling back to template-based generation."
            )
            self.faker = None
    
    def generate(
        self,
        schema: StructType,
        constraints: dict[str, TypeConstraints],
        num_rows: int = 1000,
        field_hints: dict[str, FieldHint] = None,
        auto_detect_hints: bool = True,
        null_probability: float = 0.3,
        num_partitions: Optional[int] = None
    ) -> DataFrame:
        """Generate synthetic data based on schema and constraints."""
        self._logger.info(f"Generating {num_rows} rows for {len(schema.fields)} fields")
        
        # Merge explicit hints with auto-detected
        hints = {}
        if auto_detect_hints:
            hints = self._auto_detect_hints(schema)
            self._logger.debug(f"Auto-detected {len(hints)} field hints")
        if field_hints:
            hints.update(field_hints)
            self._logger.debug(f"Total hints after merge: {len(hints)}")
        
        # Let dbldatagen decide partitioning unless user specifies
        if num_partitions:
            self._logger.debug(f"Using user-specified {num_partitions} partitions")

        # Create data generator with explicit partitioning
        datagen = self.dg.DataGenerator(
            self.spark,
            name="xsd_synthetic_data",
            rows=num_rows,
            partitions=num_partitions
        )
        
        # Configure each field
        for field in schema.fields:
            constraint = constraints.get(field.name, TypeConstraints(base_type='string'))
            hint = hints.get(field.name)
            
            datagen = self._configure_field(
                datagen, field, constraint, hint,
                null_probability if field.nullable else 0.0
            )
        
        # Build and return
        df = datagen.build()
        self._logger.info(f"Data generation complete: {num_rows} rows configured")
        
        return df
    
    def _auto_detect_hints(self, schema: StructType) -> dict[str, FieldHint]:
        """Auto-detect field hints based on field name patterns."""
        hints = {}
        
        for field in schema.fields:
            for pattern, hint in DEFAULT_FIELD_HINT_PATTERNS.items():
                if re.match(pattern, field.name, re.IGNORECASE):
                    hints[field.name] = hint
                    break
        
        return hints
    


    def _configure_field(
        self,
        datagen,
        field: StructField,
        constraint: TypeConstraints,
        hint: Optional[FieldHint],
        null_prob: float
    ):
        """Configure field for data generation based on XSD constraints."""
        name = field.name
        dtype = field.dataType
        
        # XSD constraint overrides (early returns)
        if constraint.fixed_value:
            self._logger.debug(f"Using fixed value for '{name}': '{constraint.fixed_value}'")
            return datagen.withColumn(name, "string",
                values=[constraint.fixed_value], percentNulls=0.0)
        
        if constraint.enumeration:
            self._logger.debug(f"Using enumeration for '{name}': {len(constraint.enumeration)} values")
            return datagen.withColumn(name, "string",
                values=constraint.enumeration, percentNulls=null_prob, random=True)
        
        if constraint.pattern and isinstance(dtype, StringType):
            samples = generate_pattern_samples(constraint.pattern)
            if samples:
                self._logger.debug(f"Using pattern samples for '{name}'")
                return datagen.withColumn(name, "string",
                    values=samples, percentNulls=null_prob, random=True)
        
        # Type-based dispatch
        match dtype:
            case LongType() | IntegerType():
                min_val = int(constraint.effective_min() or 0)
                max_val = int(constraint.effective_max() or 999999999)
                return datagen.withColumn(name, get_dbldatagen_type(dtype),
                    minValue=min_val, maxValue=max_val, percentNulls=null_prob)
            
            case DecimalType():
                min_val = float(constraint.effective_min() or 0)
                max_val = float(constraint.effective_max() or 999999.99)
                step = 10 ** (-dtype.scale) if dtype.scale else 0.01
                return datagen.withColumn(name, get_dbldatagen_type(dtype),
                    minValue=min_val, maxValue=max_val, step=step, percentNulls=null_prob)
            
            case DateType():
                return datagen.withColumn(name, "date", percentNulls=null_prob)
            
            case TimestampType():
                return datagen.withColumn(name, "timestamp", percentNulls=null_prob)
            
            case BooleanType():
                return datagen.withColumn(name, "boolean", percentNulls=null_prob)
            
            case StringType():
                faker_method = get_faker_method(name)
                if faker_method and self.faker:
                    self._logger.debug(f"Using Faker '{faker_method}' for '{name}'")
                    return datagen.withColumn(name, "string",
                        text=self.faker(faker_method), percentNulls=null_prob)
                return datagen.withColumn(name, "string", percentNulls=null_prob)
            
            case _:
                return datagen.withColumn(name, "string", percentNulls=null_prob)

    # XSD numeric types (for type-aware constraint priority)
    XSD_NUMERIC_TYPES = {
        'decimal', 'integer', 'int', 'long', 'short', 'byte',
        'float', 'double', 'positiveInteger', 'negativeInteger',
        'nonNegativeInteger', 'nonPositiveInteger', 'unsignedLong',
        'unsignedInt', 'unsignedShort', 'unsignedByte'
    }



print("SyntheticDataGenerator class defined with type-aware constraint priority")

## XSDLoader

In [ ]:
class XSDLoader:
    """
    Loads XSD schemas using xmlschema's native include/import resolution.
    """
    
    # Add logger instance
    _logger = logging.getLogger('xsd_synthetic.XSDLoader')
    
    def __init__(self):
        pass
    
    def load(
        self,
        xsd_path: str,
        include_paths: list[str] = None
    ) -> 'xmlschema.XMLSchema':
        """
        Load an XSD file with location overrides for includes.
        
        XSD files often have relative include paths like '../../../Common/efileTypes.xsd'
        that don't resolve correctly in different environments. The include_paths 
        parameter provides the actual locations of these files.
        
        Args:
            xsd_path: Path to the main XSD file
            include_paths: List of paths to included XSD files. These will be mapped
                          to common relative path patterns used in IRS schemas.
            
        Returns:
            Fully resolved xmlschema.XMLSchema object
        """
        self._logger.info(f"Loading XSD schema: {xsd_path}")

        # Validate files exist before loading
        missing = self.validate_paths(xsd_path, include_paths)
        if missing:
            raise FileNotFoundError(
                f"XSD file(s) not found:\n" +
                "\n".join(f"  - {p}" for p in missing) +
                "\n\nCheck your path configuration."
            )

        # Build locations dict to redirect includes to actual paths
        locations = {}
        if include_paths:
            self._logger.info(f"Setting up include path redirects for: {include_paths}")
            for path in include_paths:
                filename = os.path.basename(path)
                # Handle various relative path patterns used in IRS XSDs
                patterns = [
                    f'../../../Common/{filename}',
                    f'../../Common/{filename}',
                    f'../Common/{filename}',
                    f'Common/{filename}',
                    filename,
                ]
                for pattern in patterns:
                    locations[pattern] = path
                    self._logger.debug(f"  Mapping '{pattern}' -> '{path}'")
        
        try:
            schema = xmlschema.XMLSchema(
                xsd_path, 
                validation='lax',
                locations=locations or None
            )
            self._logger.info(
                f"Schema loaded successfully: {len(schema.types)} types, "
                f"{len(schema.elements)} elements"
            )
            return schema
        except Exception as e:
            # Try with skip validation if lax fails
            self._logger.warning(
                f"Loading with validation='lax' failed ({type(e).__name__}), "
                f"falling back to validation='skip'"
            )
            schema = xmlschema.XMLSchema(
                xsd_path, 
                validation='skip',
                locations=locations or None
            )
            self._logger.info(
                f"Schema loaded with validation='skip': {len(schema.types)} types, "
                f"{len(schema.elements)} elements"
            )
            return schema

    def get_available_elements(self, schema: 'xmlschema.XMLSchema') -> list[str]:
        """Get list of root elements available in the schema."""
        return list(schema.elements.keys())

    def get_schema_info(self, schema: 'xmlschema.XMLSchema') -> dict:
        """Get summary information about a loaded schema."""
        return {
            'target_namespace': schema.target_namespace,
            'root_elements': list(schema.elements.keys()),
            'global_types': len(schema.types),
            'attribute_groups': len(schema.attribute_groups),
        }

    def validate_paths(self, xsd_path: str, include_paths: list[str] = None) -> list[str]:
        """
        Check if XSD files exist before loading.
        
        Returns:
            List of missing file paths (empty if all exist)
        """
        missing = []
        all_paths = [xsd_path] + (include_paths or [])
        for path in all_paths:
            if not os.path.exists(path):
                missing.append(path)
        return missing

    def describe_element(self, schema: 'xmlschema.XMLSchema', element_name: str) -> dict:
        """
        Get summary of a specific element for preview.
        
        Returns dict with: field_count, type_distribution, max_depth, sample_fields
        """
        if element_name not in schema.elements:
            available = list(schema.elements.keys())[:5]
            raise ValueError(f"Element '{element_name}' not found. Available: {available}")
        
        element = schema.elements[element_name]
        
        # Count fields by walking the type tree (simplified)
        type_counts = {'string': 0, 'numeric': 0, 'boolean': 0, 'date': 0, 'other': 0}
        sample_fields = []
        
        def walk_type(xsd_type, depth=0, max_samples=10):
            if depth > 10 or xsd_type is None:
                return
            
            # Check for child elements
            if hasattr(xsd_type, 'content') and hasattr(xsd_type.content, 'iter_elements'):
                try:
                    for child in xsd_type.content.iter_elements():
                        child_name = child.name if hasattr(child, 'name') else str(child)
                        if len(sample_fields) < max_samples:
                            sample_fields.append(child_name)
                        
                        # Categorize type
                        if hasattr(child, 'type') and hasattr(child.type, 'python_type'):
                            py_type = child.type.python_type
                            if py_type == str:
                                type_counts['string'] += 1
                            elif py_type in (int, float):
                                type_counts['numeric'] += 1
                            elif py_type == bool:
                                type_counts['boolean'] += 1
                            else:
                                type_counts['other'] += 1
                        else:
                            type_counts['other'] += 1
                        
                        # Recurse
                        if hasattr(child, 'type'):
                            walk_type(child.type, depth + 1)
                except:
                    pass
        
        walk_type(element.type)
        
        return {
            'element_name': element_name,
            'field_count': sum(type_counts.values()),
            'type_distribution': {k: v for k, v in type_counts.items() if v > 0},
            'sample_fields': sample_fields,
        }


print("XSDLoader class defined (with schema discovery methods)")


## xsd_to_synthetic

In [ ]:
def xsd_to_synthetic(
    spark: SparkSession,
    xsd_path: str,
    root_element: str,
    include_paths: list[str] = None,
    num_rows: int = 1000,
    flatten_mode: str = "full",
    max_flatten_depth: int = 10,
    field_hints: dict[str, FieldHint] = None,
    auto_detect_hints: bool = True,
    null_probability: float = 0.3,
    verbose: bool = True,
    exclude_root_from_path: bool = True,
    field_separator: str = "_",
    amount_ranges: dict = None,
    date_range: tuple[str, str] = None,
    debug: str = None,
) -> DataFrame:
    """
    One-liner convenience function to generate synthetic data from an XSD.
    
    Args:
        spark: Active SparkSession
        xsd_path: Path to the main XSD file
        root_element: Name of the root element to generate data for
        include_paths: Optional list of paths to included XSD files
        num_rows: Number of synthetic rows to generate
        flatten_mode: How to handle nested types - "none", "full", or "depth:N"
        max_flatten_depth: Max depth for "depth" mode
        field_hints: Optional explicit field hints for semantic generation
        auto_detect_hints: Whether to auto-detect hints from field names
        null_probability: Probability of null values for nullable fields
        verbose: Print progress information
        exclude_root_from_path: Exclude root element name from flattened field names
        field_separator: Separator for flattened field names (e.g., "_")
        amount_ranges: Dict mapping field name patterns to (min, max) amount ranges
        date_range: Tuple of (start_date, end_date) as strings (YYYY-MM-DD)
        debug: Path to write debug log file, e.g., '/dbfs/tmp/debug.log' or '/tmp/debug.log' (default: None = no debug logging)
        
    Returns:
        DataFrame with generated synthetic data
        
    Example:
        >>> df = xsd_to_synthetic(
        ...     spark,
        ...     "IRS990.xsd",
        ...     root_element="IRS990",
        ...     include_paths=["efileTypes.xsd"],
        ...     num_rows=num_rows,
        ...     flatten_mode="full",
        ...     amount_ranges={'Revenue': (0, 10000000)},
        ...     date_range=('2020-01-01', '2024-12-31')
        ... )
    """
    
    # Add logger instance
    _logger = logging.getLogger('xsd_synthetic.xsd_to_synthetic')
    
    # Enable debug logging if requested
    if debug:
        logging.getLogger('xsd_synthetic').setLevel(logging.DEBUG)
        if debug != 'console':  # Allow file logging if path provided
            handler = logging.FileHandler(debug)
            handler.setFormatter(logging.Formatter('%(name)s - %(levelname)s - %(message)s'))
            logging.getLogger('xsd_synthetic').addHandler(handler)
        _logger.info(f"Debug logging enabled")
        print(f"Debug logging enabled{f' - writing to {debug}' if debug != 'console' else ''}")
    
    # Parse flatten mode
    if flatten_mode.startswith("depth:"):
        fm = FlattenMode.DEPTH
        max_flatten_depth = int(flatten_mode.split(":")[1])
    elif flatten_mode == "full":
        fm = FlattenMode.FULL
    elif flatten_mode == "none":
        fm = FlattenMode.NONE
    else:
        fm = FlattenMode.FULL
    
    # Log pipeline start with key parameters
    _logger.info(
        f"Starting XSD to Synthetic pipeline: xsd_path='{xsd_path}', "
        f"root_element='{root_element}', num_rows={num_rows}, "
        f"flatten_mode='{flatten_mode}', include_paths={include_paths}"
    )
    
    if verbose:
        print("=" * 60)
        print("XSD to Synthetic Data Pipeline")
        print("=" * 60)
    
    # Step 1: Load XSD
    if verbose:
        print(f"\n[1/4] Loading XSD: {xsd_path}")
        if include_paths:
            print(f"      Includes: {include_paths}")
    
    loader = XSDLoader()
    schema = loader.load(xsd_path, include_paths)
    
    if verbose:
        info = loader.get_schema_info(schema)
        print(f"      Loaded: {info['global_types']} types, "
              f"{len(info['root_elements'])} root elements")
    
    _logger.info(
        f"XSD loaded: {info['global_types']} types, "
        f"{len(info['root_elements'])} root elements"
    )
    
    # Step 2: Build Spark schema
    if verbose:
        print(f"\n[2/4] Building Spark schema for '{root_element}'")
        print(f"      Flatten mode: {fm.value}")
    
    builder = SparkSchemaBuilder(
        flatten_mode=fm,
        max_depth=max_flatten_depth,
        exclude_root_from_path=exclude_root_from_path,
        separator=field_separator
    )
    spark_schema, constraints = builder.build(schema, root_element)
    
    if verbose:
        print(f"      Generated: {len(spark_schema.fields)} fields")
        
        # Type distribution
        type_counts = {}
        for field in spark_schema.fields:
            t = type(field.dataType).__name__
            type_counts[t] = type_counts.get(t, 0) + 1
        
        print(f"      Types: {dict(sorted(type_counts.items(), key=lambda x: -x[1]))}")
    
    _logger.info(
        f"Spark schema built: {len(spark_schema.fields)} fields generated"
    )
    
    # Step 3: Generate data
    if verbose:
        print(f"\n[3/4] Generating {num_rows} synthetic rows...")
    
    generator = SyntheticDataGenerator(
        spark,
        schema=schema,  # NEW: Pass schema for xmlschema sample generation
        amount_ranges=amount_ranges,
        date_range=date_range
    )
    df = generator.generate(
        spark_schema,
        constraints,
        num_rows=num_rows,
        field_hints=field_hints,
        auto_detect_hints=auto_detect_hints,
        null_probability=null_probability
    )    # Trust dbldatagen's deterministic row generation (no count needed)
    if verbose:
        print(f"      Generated: {num_rows} rows, {len(df.columns)} columns")
    
    _logger.info(
        f"Synthetic data generated: {num_rows} rows, {len(df.columns)} columns"
    )
    
    # Step 4: Diagnostic Report
    if verbose:
        print(f"\n[4/4] Complete!")
        
        # Collect diagnostic statistics
        
        # DEBUG: Check enumeration extraction
        ind_fields_with_enum = []
        ind_fields_without_enum = []
        for field_name, constraint in constraints.items():
            if 'Ind' in field_name:
                if constraint.enumeration:
                    ind_fields_with_enum.append((field_name, constraint.enumeration))
                else:
                    ind_fields_without_enum.append(field_name)
        
        if verbose:
            print(f"\nDEBUG: Ind fields with enumeration: {len(ind_fields_with_enum)}")
            if ind_fields_with_enum:
                for fname, enum in ind_fields_with_enum[:3]:
                    print(f"  {fname}: {enum}")
            print(f"DEBUG: Ind fields without enumeration: {len(ind_fields_without_enum)}")
            if ind_fields_without_enum:
                for fname in ind_fields_without_enum[:3]:
                    print(f"  {fname}")
        
        failed_extractions = []
        default_fallbacks = []
        string_defaults = []
        
        for field_name, constraint in constraints.items():
            if constraint.extraction_failed:
                failed_extractions.append(field_name)
            if constraint.default_fallback:
                default_fallbacks.append(field_name)
            if constraint.base_type == 'string' and isinstance(constraint.spark_type, StringType):
                # Check if it's actually a string field or defaulted
                if not any(pattern in field_name for pattern in ['Nm', 'Name', 'Desc', 'Txt', 'Cd']):
                    # Don't flag fields with enumeration constraints (e.g., CheckboxType fields)
                    if not constraint.enumeration:
                        string_defaults.append(field_name)
        
        # Log diagnostic summary
        _logger.info(
            f"Diagnostic summary: {len(failed_extractions)} extraction failures, "
            f"{len(default_fallbacks)} default fallbacks, "
            f"{len(string_defaults)} potential non-string fields as string"
        )
        
        # Print diagnostic report
        if failed_extractions or default_fallbacks or string_defaults:
            print(f"\n{'='*60}")
            print("DIAGNOSTIC REPORT")
            print(f"{'='*60}")
            
            if failed_extractions:
                print(f"\nWARNING: Constraint Extraction Failures: {len(failed_extractions)}")
                print("   (Fields where facet/constraint extraction encountered errors)")
                for field in failed_extractions[:10]:  # Show first 10
                    print(f"     - {field}")
                if len(failed_extractions) > 10:
                    print(f"     ... and {len(failed_extractions) - 10} more")
            
            if default_fallbacks:
                print(f"\nWARNING: Default Fallback Fields: {len(default_fallbacks)}")
                print("   (Fields with no type information, defaulted to string)")
                for field in default_fallbacks[:10]:
                    print(f"     - {field}")
                if len(default_fallbacks) > 10:
                    print(f"     ... and {len(default_fallbacks) - 10} more")
            
            if string_defaults:
                print(f"\nWARNING: Potential Non-String Fields as String: {len(string_defaults)}")
                print("   (Fields that may be numeric/other but mapped to string)")
                for field in string_defaults[:10]:
                    print(f"     - {field}")
                if len(string_defaults) > 10:
                    print(f"     ... and {len(string_defaults) - 10} more")
            
            # Summary statistics
            total_issues = len(set(failed_extractions + default_fallbacks + string_defaults))
            print(f"\nTotal fields with potential issues: {total_issues}/{len(constraints)} ({total_issues/len(constraints)*100:.1f}%)")
            print(f"Clean fields: {len(constraints) - total_issues}/{len(constraints)} ({(len(constraints)-total_issues)/len(constraints)*100:.1f}%)")
        else:
            print("\nNo diagnostic issues detected - all constraints extracted successfully!")
        
        print("=" * 60)
    
    _logger.info("Pipeline complete")
    
    return df


def explore_xsd(
    path: str,
    include_paths: list[str] = None,
    root_element: str = None,
    quiet: bool = True
) -> None:
    """
    Explore an XSD file or directory without generating data.
    
    Use this to understand an unfamiliar XSD package before running full generation.
    Accepts either a single XSD file or a directory containing multiple XSD files.
    
    Args:
        path: Path to XSD file OR directory containing XSD files
        include_paths: Optional list of paths to included XSD files (only for single file mode)
        root_element: Optional root element to preview (if not provided, lists all available)
        quiet: Suppress xmlschema warnings about missing includes (default: True)
    
    Example:
        >>> explore_xsd("/path/to/990T Schema 2024/")  # Scan entire directory
        >>> explore_xsd("/path/to/IRS990T.xsd")  # Single file
        >>> explore_xsd("/path/to/IRS990T.xsd", ["/path/to/efileTypes.xsd"], "IRS990T")
    """
    from pathlib import Path
    import glob
    
    path_obj = Path(path)
    
    # Check if it's a directory
    if path_obj.is_dir():
        _explore_xsd_directory(path_obj, root_element, quiet)
    else:
        _explore_xsd_file(path, include_paths, root_element)


def _explore_xsd_directory(dir_path, root_element: str = None, quiet: bool = True) -> None:
    """Scan a directory for XSD files and summarize them."""
    from pathlib import Path
    
    print(f"\n{'='*60}")
    print("XSD Package Explorer")
    print(f"{'='*60}")
    print(f"\nScanning: {dir_path}")
    
    # Find all XSD files recursively
    xsd_files = sorted(dir_path.rglob("*.xsd"))
    
    if not xsd_files:
        print(f"\n[ERROR] No .xsd files found in {dir_path}")
        return
    
    print(f"Found {len(xsd_files)} XSD files\n")
    
    # Analyze each file
    schemas_with_roots = []
    type_only_files = []
    failed_files = []
    
    loader = XSDLoader()
    
    # Suppress warnings during directory scan (lots of "missing include" noise)
    import warnings
    from contextlib import nullcontext
    warning_filter = warnings.catch_warnings() if quiet else nullcontext()
    
    with warning_filter:
        if quiet:
            warnings.filterwarnings('ignore')  # Suppress all warnings during scan
        
        for xsd_file in xsd_files:
            rel_path = xsd_file.relative_to(dir_path)
            try:
                # Try to load schema
                schema = loader.load(str(xsd_file))
                elements = loader.get_available_elements(schema)
                info = loader.get_schema_info(schema)
                
                if elements:
                    schemas_with_roots.append({
                        'path': rel_path,
                        'full_path': xsd_file,
                        'elements': elements,
                        'types': info['global_types']
                    })
                else:
                    type_only_files.append({
                        'path': rel_path,
                        'types': info['global_types']
                    })
            except Exception as e:
                failed_files.append({
                    'path': rel_path,
                    'error': str(e)[:50]
                })
    
    # Identify likely "main" forms vs supporting schedules
    def is_likely_main_form(element_name: str) -> bool:
        """Heuristic: main forms are IRS#### without Schedule/Statement/etc."""
        skip_patterns = ['Schedule', 'Statement', 'Attachment', 'Manifest', 
                         'Header', 'Receipt', 'Acknowledgement', 'Binary']
        if any(p in element_name for p in skip_patterns):
            return False
        # IRS forms: IRS990, IRS990T, IRS1120, etc.
        import re
        if re.match(r'^IRS\d+[A-Z]?$', element_name):
            return True
        # Return element is the full submission wrapper
        if element_name == 'Return':
            return True
        return False
    
    # Collect main form candidates
    main_forms = []
    for s in schemas_with_roots:
        for elem in s['elements']:
            if is_likely_main_form(elem):
                main_forms.append({
                    'element': elem,
                    'file': s['path'],
                    'full_path': s['full_path']
                })
    
    # Show suggested main forms first
    if main_forms:
        # Deduplicate by element name, keeping first occurrence
        seen = set()
        unique_main = []
        for m in main_forms:
            if m['element'] not in seen:
                seen.add(m['element'])
                unique_main.append(m)
        
        print(f"{'─'*60}")
        print(f"SUGGESTED MAIN FORMS ({len(unique_main)} found)")
        print(f"{'─'*60}")
        print("  (These look like primary IRS forms, not schedules/attachments)")
        for m in unique_main[:15]:
            print(f"    - {m['element']}  ({m['file']})")
        if len(unique_main) > 15:
            print(f"    ... and {len(unique_main) - 15} more")
    
    # Print schemas with root elements (the interesting ones)
    if schemas_with_roots:
        print(f"\n{'─'*60}")
        print(f"ALL SCHEMAS WITH ROOT ELEMENTS ({len(schemas_with_roots)} files)")
        print(f"{'─'*60}")
        for s in schemas_with_roots:
            print(f"\n  {s['path']}")
            print(f"    Types: {s['types']}, Root elements: {len(s['elements'])}")
            for elem in s['elements'][:5]:
                print(f"      - {elem}")
            if len(s['elements']) > 5:
                print(f"      ... and {len(s['elements']) - 5} more")
    
    # Print type-only files (supporting files)
    if type_only_files:
        print(f"\n{'─'*60}")
        print(f"TYPE DEFINITION FILES ({len(type_only_files)} files)")
        print(f"{'─'*60}")
        print("  (These define types but have no root elements)")
        for t in type_only_files[:10]:
            print(f"    {t['path']} ({t['types']} types)")
        if len(type_only_files) > 10:
            print(f"    ... and {len(type_only_files) - 10} more")
    
    # Print failures
    if failed_files:
        print(f"\n{'─'*60}")
        print(f"FAILED TO LOAD ({len(failed_files)} files)")
        print(f"{'─'*60}")
        for f in failed_files[:5]:
            print(f"    {f['path']}: {f['error']}")
        if len(failed_files) > 5:
            print(f"    ... and {len(failed_files) - 5} more")
    
    # If root element specified, find it and preview
    if root_element:
        found = False
        for s in schemas_with_roots:
            if root_element in s['elements']:
                print(f"\n{'─'*60}")
                print(f"PREVIEW: {root_element}")
                print(f"{'─'*60}")
                print(f"Found in: {s['path']}")
                
                # Load and preview
                schema = loader.load(str(s['full_path']))
                builder = SparkSchemaBuilder(
                    flatten_mode=FlattenMode.FULL,
                    max_depth=10,
                    exclude_root_from_path=True
                )
                builder.preview(schema, root_element)
                found = True
                break
        
        if not found:
            print(f"\n[WARNING] Root element '{root_element}' not found in any schema")
    else:
        # Suggest next steps - prefer main forms if found
        if main_forms or schemas_with_roots:
            print(f"\n{'─'*60}")
            print("NEXT STEPS")
            print(f"{'─'*60}")
            
            # Use first main form if available, otherwise first element
            if main_forms:
                suggested = main_forms[0]
                suggested_elem = suggested['element']
                suggested_path = suggested['full_path']
            else:
                first_schema = schemas_with_roots[0]
                suggested_elem = first_schema['elements'][0]
                suggested_path = first_schema['full_path']
            
            print(f"  To preview a specific element:")
            print(f"    explore_xsd('{dir_path}', root_element='{suggested_elem}')")
            print(f"\n  To generate data:")
            print(f"    df = xsd_to_synthetic(")
            print(f"        spark,")
            print(f"        xsd_path='{suggested_path}',")
            print(f"        root_element='{suggested_elem}',")
            print(f"        num_rows=1000")
            print(f"    )")
    
    print(f"\n{'='*60}\n")


def _explore_xsd_file(xsd_path: str, include_paths: list[str], root_element: str) -> None:
    """Explore a single XSD file."""
    print(f"\n{'='*60}")
    print("XSD Explorer")
    print(f"{'='*60}")
    
    # Load schema
    loader = XSDLoader()
    
    # Check for missing files first
    missing = loader.validate_paths(xsd_path, include_paths)
    if missing:
        print(f"\n[ERROR] Missing files:")
        for p in missing:
            print(f"  - {p}")
        return
    
    print(f"\nLoading: {xsd_path}")
    if include_paths:
        print(f"Includes: {include_paths}")
    
    try:
        schema = loader.load(xsd_path, include_paths)
    except Exception as e:
        print(f"\n[ERROR] Failed to load schema: {e}")
        return
    
    # Show schema info
    info = loader.get_schema_info(schema)
    print(f"\nSchema Summary:")
    print(f"  Global types: {info['global_types']}")
    print(f"  Root elements: {len(info['root_elements'])}")
    
    # List available elements
    elements = loader.get_available_elements(schema)
    print(f"\nAvailable Root Elements:")
    for elem in elements[:20]:
        print(f"  - {elem}")
    if len(elements) > 20:
        print(f"  ... and {len(elements) - 20} more")
    
    # If root element specified, show preview
    if root_element:
        builder = SparkSchemaBuilder(
            flatten_mode=FlattenMode.FULL,
            max_depth=10,
            exclude_root_from_path=True
        )
        builder.preview(schema, root_element)
    else:
        print(f"\nTip: Call explore_xsd(..., root_element='ElementName') to preview a specific element")
    
    print(f"{'='*60}\n")


print("xsd_to_synthetic() and explore_xsd() convenience functions defined")

In [ ]:
# =============================================================================
# Unity Catalog & Databricks Integration (2025 Best Practices)
# =============================================================================

class DatabricksRuntime:
    """Detect and utilize Databricks-specific features."""
    
    _logger = logging.getLogger('xsd_synthetic.DatabricksRuntime')
    
    @staticmethod
    def is_databricks() -> bool:
        """Check if running on Databricks."""
        try:
            import dbutils  # noqa
            return True
        except (ImportError, NameError):
            return False
    
    @staticmethod
    def get_runtime_version() -> Optional[str]:
        """Get Databricks runtime version."""
        if not DatabricksRuntime.is_databricks():
            return None
        
        try:
            from pyspark.sql import SparkSession
            spark = SparkSession.getActiveSession()
            if spark:
                return spark.conf.get("spark.databricks.clusterUsageTags.sparkVersion")
        except Exception:
            pass
        
        return None
    
    @staticmethod
    def supports_liquid_clustering() -> bool:
        """Check if liquid clustering is supported (DBR 14.0+)."""
        version = DatabricksRuntime.get_runtime_version()
        if version:
            try:
                major = int(version.split('.')[0])
                return major >= 14
            except (ValueError, AttributeError):
                pass
        return False


class UnityCatalogWriter:
    """Write to Unity Catalog with liquid clustering and optimizations."""
    
    _logger = logging.getLogger('xsd_synthetic.UnityCatalogWriter')
    
    @staticmethod
    def write_synthetic_data(
        df,  # DataFrame
        catalog: str,
        schema: str,
        table: str,
        mode: str = "overwrite",
        cluster_by: Optional[list] = None,
        enable_optimizations: bool = True,
        table_comment: Optional[str] = None
    ) -> str:
        """Write DataFrame to Unity Catalog. Returns full table name."""
        full_table_name = f"{catalog}.{schema}.{table}"
        
        UnityCatalogWriter._logger.info(
            f"Writing to Unity Catalog: {full_table_name} (mode={mode})"
        )
        
        # Build writer with Delta format
        writer = df.write.format("delta").mode(mode)
        
        # Add table comment
        if table_comment:
            writer = writer.option("comment", table_comment)
        
        # Enable liquid clustering (2025 best practice)
        if cluster_by:
            if DatabricksRuntime.supports_liquid_clustering():
                UnityCatalogWriter._logger.info(
                    f"Enabling liquid clustering on columns: {cluster_by}"
                )
                writer = writer.option("clusterBy", ",".join(cluster_by))
            else:
                UnityCatalogWriter._logger.warning(
                    f"Liquid clustering not supported on this runtime. "
                    f"Requires DBR 14.0+. Consider upgrading."
                )
        
        # Enable optimizations
        if enable_optimizations:
            UnityCatalogWriter._logger.info("Enabling optimized writes and auto-compaction")
            writer = (
                writer
                .option("delta.autoOptimize.optimizeWrite", "true")
                .option("delta.autoOptimize.autoCompact", "true")
            )
        
        # Write to Unity Catalog
        try:
            writer.saveAsTable(full_table_name)
            UnityCatalogWriter._logger.info(f"Successfully wrote to {full_table_name}")
        except Exception as e:
            UnityCatalogWriter._logger.error(f"Failed to write to {full_table_name}: {e}")
            raise
        
        # Enable predictive optimization at table level
        if enable_optimizations:
            try:
                spark = df.sparkSession
                spark.sql(
                    f"ALTER TABLE {full_table_name} "
                    f"SET TBLPROPERTIES ('delta.autoOptimize.autoCompact' = 'true', "
                    f"'delta.autoOptimize.optimizeWrite' = 'true')"
                )
                UnityCatalogWriter._logger.info(
                    f"Predictive optimization enabled for {full_table_name}"
                )
            except Exception as e:
                UnityCatalogWriter._logger.warning(
                    f"Could not enable predictive optimization: {e}. "
                    f"This may require additional permissions."
                )
        
        return full_table_name


print("Unity Catalog utilities defined")


In [ ]:
# Cell intentionally left empty (logging helpers removed - use enable_debug() or standard logging module)
